In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [2]:
datasetPath = Path("/home/philipm/Desktop/ML-For-CV-Robustness/datasets/shadows-detection-dataset/trees-dl-shadows.csv")

#making sure
datasetPath.is_file()

True

In [3]:
dataset = pd.read_csv(datasetPath)
dataset.head()

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut,shadows
0,0.248512,0.000147,0.000204,1136.471,1031.228,0.0,180.0,-71.945084,19.489212,0.0,-19.35,-0.0,0.143482,0.0
1,0.209356,0.000254,0.000347,1136.471,1031.228,1.0,180.0,-71.945084,19.489212,0.0,-19.35,-0.0,0.141526,0.0
2,0.228622,0.000229,0.000317,1136.471,1031.228,2.0,180.0,-71.945084,19.489212,0.0,-19.35,-0.0,0.139981,0.0
3,0.193133,0.000223,0.000293,1136.471,1031.228,3.0,180.0,-71.945084,19.489212,0.0,-19.35,-0.0,0.133106,0.0
4,0.258888,0.000187,0.000269,1136.471,1031.228,4.0,180.0,-71.945084,19.489212,0.0,-19.35,-0.0,0.145066,0.0


In [4]:
target_col = dataset["shadows"]
labels = target_col.to_numpy(dtype="float32")
labels

array([0., 0., 0., ..., 0., 0., 0.], shape=(332640,), dtype=float32)

In [5]:
inputs = dataset.drop(["sunRoll", "camRoll", "grviOut", "shadows"], axis=1)
inputs = inputs.to_numpy(dtype="float32")
inputs

array([[ 2.4851234e-01,  1.4691126e-04,  2.0387112e-04, ...,
         1.9489212e+01, -1.9350000e+01, -0.0000000e+00],
       [ 2.0935635e-01,  2.5367978e-04,  3.4656093e-04, ...,
         1.9489212e+01, -1.9350000e+01, -0.0000000e+00],
       [ 2.2862221e-01,  2.2864803e-04,  3.1710259e-04, ...,
         1.9489212e+01, -1.9350000e+01, -0.0000000e+00],
       ...,
       [ 8.0871526e-03,  3.2670033e-01,  3.1690043e-01, ...,
         1.0435840e+02, -2.2218000e+01, -1.8000000e+02],
       [ 1.2527064e-02,  2.7300212e-01,  2.6233262e-01, ...,
         1.0435840e+02, -2.2218000e+01, -1.8000000e+02],
       [ 2.2494195e-02,  2.6528352e-01,  2.5932705e-01, ...,
         1.0435840e+02, -2.2218000e+01, -1.8000000e+02]],
      shape=(332640, 10), dtype=float32)

In [6]:
if len(labels) != len (inputs):
    print ("problem")

In [7]:
n_total = len(inputs)

n_train = int(0.8*n_total)
n_test = int(0.1*n_total)
n_val = int(0.1*n_total)

# shuffle
g = torch.Generator().manual_seed(42)
perm = torch.randperm(n_total, generator=g).numpy()

inputs_shuf = inputs[perm]
labels_shuf = labels[perm]

# split
train_inputs, train_labels = inputs_shuf[:n_train], labels_shuf[:n_train]
val_inputs, val_labels = inputs_shuf[n_train:n_train + n_val], labels_shuf[n_train:n_train + n_val]
test_inputs, test_labels = inputs_shuf[n_train + n_val:], labels_shuf[n_train + n_val:]

In [8]:
# scaling
from sklearn.preprocessing import QuantileTransformer, StandardScaler

if True:
    input_scaler = StandardScaler() #QuantileTransformer(output_distribution='normal')
    target_scaler = StandardScaler()

    train_inputs = input_scaler.fit_transform(train_inputs)
    #train_labels = target_scaler.fit_transform(train_labels.reshape(-1, 1)).reshape(-1)

    val_inputs = input_scaler.transform(val_inputs)
    #val_labels = target_scaler.transform(val_labels.reshape(-1, 1)).reshape(-1)

    test_inputs = input_scaler.transform(test_inputs)
    #test_labels = target_scaler.transform(test_labels.reshape(-1, 1)).reshape(-1)

In [9]:
train_inputs = torch.from_numpy(train_inputs).float()
train_labels = torch.from_numpy(train_labels).float()

val_inputs = torch.from_numpy(val_inputs).float()
val_labels = torch.from_numpy(val_labels).float()

test_inputs = torch.from_numpy(test_inputs).float()
test_labels = torch.from_numpy(test_labels).float()

In [10]:
print ("Train size:", len(train_inputs))
print ("Val size:", len(val_inputs))
print ("Test size:", len(test_inputs))

Train size: 266112
Val size: 33264
Test size: 33264


In [11]:
BATCH_SIZE = 512

class metricsDataset (Dataset):
    def __init__(self, metrics, labels):
        self.metrics = metrics
        self.labels = labels

    def __len__(self):
        return len(self.metrics)

    def __getitem__(self, idx):
        return self.metrics[idx], self.labels[idx]

train_dataset = metricsDataset(train_inputs, train_labels)
val_dataset   = metricsDataset(val_inputs, val_labels)
test_dataset  = metricsDataset(test_inputs, test_labels)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    pin_memory=True
)

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cuda


In [13]:
model = nn.Sequential(
    nn.Linear(10, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),

    nn.Linear(64, 32),
    nn.ReLU(),

    nn.Linear(32, 1)
).to(device)

In [14]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=0.01
)

In [15]:
EPOCHS = 160

best_val_loss = float("inf")

for epoch in range(EPOCHS):

    model.train()
    train_loss = 0.0

    for x, y in train_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True).float()

        optimizer.zero_grad()

        logits = model(x).squeeze(-1)

        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * x.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True).float()

            logits = model(x).squeeze(-1)

            loss = criterion(logits, y)

            val_loss += loss.item() * x.size(0)

    val_loss /= len(val_loader.dataset)

    print(
        f"Epoch {epoch + 1:3d}/{EPOCHS} | "
        f"Train BCE: {train_loss:.6f} | "
        f"Val BCE: {val_loss:.6f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_loss": best_val_loss,
        }, "shadowsClassifier.pth")

        print("Best Model Saved")

Epoch   1/160 | Train BCE: 0.462632 | Val BCE: 0.436296
Best Model Saved
Epoch   2/160 | Train BCE: 0.413940 | Val BCE: 0.392570
Best Model Saved
Epoch   3/160 | Train BCE: 0.372512 | Val BCE: 0.375163
Best Model Saved
Epoch   4/160 | Train BCE: 0.361303 | Val BCE: 0.391044
Epoch   5/160 | Train BCE: 0.356604 | Val BCE: 0.377359
Epoch   6/160 | Train BCE: 0.352299 | Val BCE: 0.414136
Epoch   7/160 | Train BCE: 0.350596 | Val BCE: 0.382327
Epoch   8/160 | Train BCE: 0.349766 | Val BCE: 0.376487
Epoch   9/160 | Train BCE: 0.349220 | Val BCE: 0.389335
Epoch  10/160 | Train BCE: 0.347016 | Val BCE: 0.371025
Best Model Saved
Epoch  11/160 | Train BCE: 0.347225 | Val BCE: 0.359841
Best Model Saved
Epoch  12/160 | Train BCE: 0.346639 | Val BCE: 0.389011
Epoch  13/160 | Train BCE: 0.344856 | Val BCE: 0.357722
Best Model Saved
Epoch  14/160 | Train BCE: 0.346144 | Val BCE: 0.368754
Epoch  15/160 | Train BCE: 0.343292 | Val BCE: 0.359433
Epoch  16/160 | Train BCE: 0.344295 | Val BCE: 0.329977
Be

In [16]:
checkpoint = torch.load(
    "shadowsClassifier.pth",
    map_location=device
)
model.load_state_dict(checkpoint["model_state_dict"])
print(
    f"Loaded best model "
    f"(Val BCE: {checkpoint['best_val_loss']:.6f})"
)

model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for x, y in test_loader:

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True).float()

        logits = model(x).squeeze(-1)

        loss = criterion(logits, y)
        test_loss += loss.item() * x.size(0)

        probabilities = torch.sigmoid(logits)

        predictions = (probabilities >= 0.5).float()

        correct += (predictions == y).sum().item()
        total += y.numel()

test_loss /= total
accuracy = 100.0 * correct / total

print(f"\nTest BCE Loss: {test_loss:.6f}")
print(f"Test Accuracy: {accuracy:.2f}%")
print(f"Correct: {correct}/{total}")

Loaded best model (Val BCE: 0.311667)

Test BCE Loss: 0.315188
Test Accuracy: 87.23%
Correct: 29015/33264
